In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import Libraries

In [ ]:
pip install sentence-transformers transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np
import string

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer
from datasets import Dataset


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# EDA

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
train.head()

In [ ]:
train['answer'].value_counts()

In [ ]:
train.isnull().sum()

# Evaluation Metric

In [ ]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Model 2 (Pretrained): DeBERTa

In [ ]:
train_questions, val_questions = train_test_split(
    train,
    test_size=0.2,
    stratify=train["answer"],
    random_state=4524
)

print(train_questions.shape)
print(val_questions.shape)

In [ ]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_multiple_choice(examples):
    # Repeat the prompt 5 times for the 5 choices
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    
    # Grab options A through E for each question
    choices = ["A", "B", "C", "D", "E"]
    second_sentences = [
        [examples[choice][i] for choice in choices]
        for i in range(len(examples["prompt"]))
    ]
    
    # Flatten both lists to pass to the tokenizer
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    # Tokenize pairs
    tokenized = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=384, 
        padding="max_length"
    )
    
    # Unflatten back into the shape: (batch_size, num_choices, seq_len)
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

# Convert string answers (A-E) to integers (0-4)
choice_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# Map directly from your original splits
train_questions["label"] = train_questions["answer"].map(choice_map)
val_questions["label"] = val_questions["answer"].map(choice_map)

train_ds = Dataset.from_pandas(train_questions)
val_ds = Dataset.from_pandas(val_questions)

# Tokenize using the new function
train_ds = train_ds.map(preprocess_multiple_choice, batched=True)
val_ds = val_ds.map(preprocess_multiple_choice, batched=True)

In [ ]:
# UPDATE THIS CELL TO MATCH THE NEW STRUCTURE
train_ds = train_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])
val_ds = val_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])

# Note the column name 'label' here (no 's')
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(model_name,torch_dtype=torch.float32)
model = model.cuda()

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=2,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  
    gradient_checkpointing=True,    
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                      
    bf16=False,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [ ]:
trainer.train()

In [ ]:
choices = ["A", "B", "C", "D", "E"]

def predict_question(row):
    # Format inputs exactly like training
    first_sentences = [row["prompt"]] * 5
    second_sentences = [row[c] for c in choices]
    
    # Match the max_length padding used during training
    encodings = tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=384,
        return_tensors="pt"
    )
    
    # Reshape tensors to add the batch dimension: (1, 5, seq_len)
    encodings = {k: v.unsqueeze(0).to(model.device) for k, v in encodings.items()}
    
    with torch.no_grad():
        outputs = model(**encodings)
        # Outputs shape is (1, 5). Grab the logits directly.
        logits = outputs.logits.detach().cpu().numpy()[0]
        
    # Sort options by highest logit score descending
    order = np.argsort(logits)[::-1]
    return [choices[i] for i in order[:3]]

In [ ]:
val_predictions = []

for _, row in val_questions.iterrows():

    val_predictions.append(
        predict_question(row)
    )

score = map3(
    val_questions["answer"].tolist(),
    val_predictions
)

print("Validation MAP@3:", score)

In [ ]:
test_predictions = []

for _, row in test.iterrows():

    test_predictions.append(
        " ".join(
            predict_question(row)
        )
    )

# Submission Cell

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()